# Atividade de separatrizes: PIB per capita dos municípios do RN

Este notebook **responde a atividade**, pergunta por pergunta. Cada bloco de **texto** explica a ideia em português simples. Cada bloco de **código** faz o cálculo. Você **não precisa saber Python**: leia o texto e, no código, leia as linhas que começam com `#` (o Python ignora essas linhas; elas são só explicação).

Há **dois notebooks** neste projeto. Use nesta ordem:

1. `gerar_dataset.ipynb` — **baixa** os dados do IBGE e grava a pasta `data/` (só precisa rodar uma vez).
2. **Este arquivo** — **analisa** os dados e responde as perguntas 1, 2.1, 2.2, 2.3 e a descrição.

Se a pasta `data/` ainda não existir, a célula de dados deste notebook tenta baixar do IBGE sozinha (precisa da pasta `src/` e de internet). **CSV não é obrigatório.**

## O que a atividade pede

1. Obter o **PIB per capita** de **todos os municípios do Rio Grande do Norte**.
2. Criar o grupo com os **10% municípios de menor PIB per capita**.
   - **2.1** Calcular o **percentil 10** (a separatriz).
   - **2.2** Separar os municípios com PIB per capita **menor que** o percentil 10.
   - **2.3** Apresentar variáveis que possam explicar esses PIBs per capita.
3. **Descrever sumariamente** essa situação.

## Como executar

1. No menu: **Ambiente de execução → Executar tudo** (Colab) ou **Run All** (VS Code / Jupyter).
2. Espere a primeira célula instalar bibliotecas (no Colab isso é normal).
3. Desça a página: o resultado de cada pergunta aparece **logo abaixo** do código.

### Google Colab

Funciona no Colab, mas **não envie só este arquivo**. O notebook precisa da pasta `src/` (código que fala com o IBGE).

Jeito mais simples: publique o repositório no GitHub e, no Colab, rode:

```
!git clone https://github.com/AndressaLF/PIB_Munincipios_RN.git
%cd PIB_Munincipios_RN
```

Depois abra `atividade_separatrizes_pib_rn.ipynb` **dentro** da pasta clonada.

Portal do IBGE (página, não é API): [Cidades@ — Panorama do RN](https://cidades.ibge.gov.br/brasil/rn/panorama).


## Mini glossário (para quem não programa)

| Palavra | Significado aqui |
| --- | --- |
| **Notebook** | Este arquivo. Mistura texto (explicação) e código (cálculo). |
| **Célula** | Um bloco. Células de texto explicam. Células de código calculam. |
| **Biblioteca** | Pacote pronto de funções. Exemplo: `pandas` trabalha com tabelas. |
| **DataFrame** | Nome que o Python dá para uma **tabela** (linhas = municípios, colunas = indicadores). |
| **Variável** | Uma caixinha com um nome que guarda um valor. Exemplo: `percentil_10` guarda o valor do P10. |
| **Função** | Um comando que recebe dados e devolve um resultado. Exemplo: `mean()` calcula a média. |
| **API** | Um endereço na internet que devolve dados (em vez de uma página bonita). O IBGE tem APIs oficiais. |
| **PIB per capita** | PIB do município dividido pela população. É uma média de riqueza produzida por habitante, não a renda de cada pessoa. |
| **Separatriz / percentil 10** | Valor que deixa cerca de **10%** dos municípios com PIB per capita **abaixo** dele. |

No código, tudo que vem depois de `#` é só explicação: o Python ignora.


## 0. Preparar as ferramentas

Vamos usar três bibliotecas:

- **pandas**: lê e organiza tabelas.
- **numpy**: faz o cálculo do percentil.
- **pathlib**: encontra a pasta dos arquivos neste computador.

A célula abaixo só **carrega** essas ferramentas. Ela não baixa os dados do IBGE ainda.


In [ ]:
# Esta linha instala as bibliotecas. No Colab é obrigatório na primeira vez.
# No computador, se já instalou com pip install -r requirements.txt, ela só confirma.
%pip install -q pandas numpy openpyxl matplotlib

import os          # pastas e comandos do sistema (ex.: mudar de diretório)
import sys         # caminho de importação dos arquivos da pasta src/
import pandas as pd   # pandas = planilha dentro do Python (tabela com nome de colunas)
import numpy as np    # numpy = contas estatísticas (aqui: o percentil 10)
from pathlib import Path  # Path = endereço de arquivo de forma organizada

# Números na tela no padrão brasileiro: 13668.37 aparece como 13.668,37
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

# IN_COLAB fica True só se este notebook estiver rodando na nuvem do Google
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Ferramentas carregadas com sucesso.")
print("Rodando no Google Colab:", "sim" if IN_COLAB else "não")


## 0.1 Uma função só para escrever reais no formato brasileiro

No Python, milhares usam vírgula americana (`13,668.37`). No Brasil escrevemos `13.668,37`. A função abaixo só formata o número para a leitura ficar mais fácil. Ela **não muda** o cálculo, só a apresentação.


In [ ]:
def br(valor, casas=2):
    """Escreve um número no formato brasileiro, só para leitura.

    Exemplo: 13668.37  vira o texto  "13.668,37"
    Isso NÃO altera o cálculo, só o que aparece na tela.
    """
    # Se o valor estiver vazio, mostramos um traço
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return "-"
    # Primeiro o Python formata no estilo americano (vírgula nos milhares)
    texto = f"{float(valor):,.{casas}f}"
    # Troca para o estilo brasileiro: ponto nos milhares e vírgula nos centavos
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


print("Exemplo de formatação (não é um dado do IBGE):", br(13668.37))


## 0.2 De onde vêm os dados?

O site [cidades.ibge.gov.br](https://cidades.ibge.gov.br/brasil/rn/panorama) **não é uma API**. É uma página. Por trás dela, o IBGE publica APIs oficiais (endereços que devolvem JSON).

Este projeto consulta:

1. **API de Localidades** — lista os 167 municípios do RN (código da UF = 24).
2. **API de Pesquisas (Cidades@)** — PIB per capita 2023 e valor adicionado por setor em 2021.
3. **API de Agregados (SIDRA)** — população, área, densidade e alfabetização do Censo 2022.

O recorte `N6[N3[24]]` significa: *todos os municípios (N6) que estão dentro do estado 24 (RN)*.

**CSV não é obrigatório.** A célula seguinte tenta, nesta ordem:

1. ler `data/municipios_rn.csv`, se já existir (mais rápido);
2. no **Colab**, clonar o GitHub para achar a pasta `src/`;
3. se `src/` existir, **baixar do IBGE** (internet, cerca de 1 minuto).


In [ ]:
# Path.cwd() = pasta em que o notebook está rodando agora
raiz = Path.cwd()
REPO_GITHUB = "https://github.com/AndressaLF/PIB_Munincipios_RN.git"


def achar_projeto(pasta):
    """Procura a pasta do projeto (precisa ter data/*.csv ou src/pipeline.py)."""
    candidatas = [pasta, pasta.parent, pasta / "PIB_Munincipios_RN", Path("/content/PIB_Munincipios_RN")]
    for c in candidatas:
        if (c / "data" / "municipios_rn.csv").exists() or (c / "src" / "pipeline.py").exists():
            return c
    return pasta


# No Colab o arquivo começa em /content. Se src/ não estiver lá, clonamos o GitHub.
if IN_COLAB and not (raiz / "src" / "pipeline.py").exists():
    destino = Path("/content/PIB_Munincipios_RN")
    if not (destino / "src" / "pipeline.py").exists():
        print("Clonando o repositório no Colab (o código src/ precisa estar no GitHub)...")
        os.system(f"git clone {REPO_GITHUB} {destino}")
    if (destino / "src" / "pipeline.py").exists():
        os.chdir(destino)  # entra na pasta do projeto
        raiz = destino
        print("Pasta do projeto no Colab:", raiz)

raiz = achar_projeto(Path.cwd())
os.chdir(raiz)
arquivo_csv = raiz / "data" / "municipios_rn.csv"
print("Procurando o CSV:", arquivo_csv)

if arquivo_csv.exists():
    # Atalho: usa o arquivo já gerado (não consulta o IBGE de novo)
    tabela = pd.read_csv(arquivo_csv)
    tabela["abaixo_percentil_10"] = tabela["abaixo_percentil_10"].astype(str).isin(["True", "true", "1"])
    print("Dados lidos do CSV local. CSV não é obrigatório; é só mais rápido.")
elif (raiz / "src" / "pipeline.py").exists():
    # Sem CSV: baixa do IBGE (precisa de internet, cerca de 1 minuto)
    print("CSV não encontrado. Baixando do IBGE...")
    sys.path.insert(0, str(raiz))
    from src.pipeline import executar
    tabela, resumo_coleta, arquivos = executar()
    print("Coleta concluída. Arquivos em:", arquivos)
else:
    raise FileNotFoundError(
        "Faltou a pasta src/. No Colab, clone o repositório completo no GitHub "
        "ou envie a pasta do projeto, não apenas o arquivo .ipynb. "
        "O CSV é opcional; o código em src/ é que fala com o IBGE."
    )

print("Quantidade de municípios (linhas da tabela):", tabela.shape[0])
print("Quantidade de variáveis (colunas da tabela):", tabela.shape[1])


### O que cada coluna da tabela significa

| Coluna | O que é |
| --- | --- |
| `municipio` | Nome da cidade |
| `pib_per_capita` | PIB por habitante em 2023, em reais |
| `pib_mil_reais` | PIB total de 2023, em mil reais |
| `populacao_censo_2022` | Habitantes no Censo 2022 |
| `densidade_demografica` | Habitantes por km² |
| `participacao_adm_publica_pct` | Quanto da economia local vem da administração pública (2021) |
| `participacao_industria_pct` | Peso da indústria no valor adicionado (2021) |
| `taxa_alfabetizacao_15_mais_pct` | % de pessoas com 15 anos ou mais que sabem ler e escrever |
| `idhm` | Índice de Desenvolvimento Humano Municipal |


---
# Pergunta 1. PIB per capita de todos os municípios do RN

**O que vamos fazer:** olhar a coluna `pib_per_capita` para os 167 municípios.

**Por que isso importa:** sem a lista completa não dá para calcular o percentil 10. O P10 é uma posição **dentro da distribuição de todos** os municípios, não de um recorte escolhido à mão.


In [ ]:
# Copia só as colunas desta pergunta (não apaga a tabela original)
pib_todos = tabela[
    ["codigo_municipio", "municipio", "mesorregiao", "microrregiao", "pib_per_capita", "pib_mil_reais"]
].copy()

# ascending=True = do menor PIB per capita para o maior
pib_todos = pib_todos.sort_values("pib_per_capita", ascending=True)

# ranking 1 = município com o menor PIB per capita do estado
pib_todos["ranking"] = range(1, len(pib_todos) + 1)

print("Total de municípios do RN nesta tabela:", len(pib_todos))
print()
print("Menor PIB per capita (R$):", br(pib_todos["pib_per_capita"].min()))
print("Maior PIB per capita (R$):", br(pib_todos["pib_per_capita"].max()))
print("Média do RN (R$):", br(pib_todos["pib_per_capita"].mean()))
print("Mediana do RN (R$):", br(pib_todos["pib_per_capita"].median()))
print()
print("A mediana é o valor do meio da fila.")
print("Se a média fica bem acima da mediana, poucos municípios muito ricos puxam a média.")


### Resposta da pergunta 1

Abaixo está a lista completa, já ordenada. Role a tabela para ver os 167 municípios.

O ano do PIB é **2023** (série revisada do IBGE, pesquisa 38, indicador 47001).


In [ ]:
# Mostra a tabela inteira na tela
pib_todos


Os 5 menores e os 5 maiores ajudam a ver a desigualdade entre municípios:


In [ ]:
# head(5) = as 5 primeiras linhas da tabela já ordenada = os 5 menores PIBs
print("5 municípios com MENOR PIB per capita")
display(pib_todos.head(5))

# tail(5) = as 5 últimas linhas = os 5 maiores PIBs
print("5 municípios com MAIOR PIB per capita")
display(pib_todos.tail(5))


---
# Pergunta 2. Grupo com os 10% de menor PIB per capita

**Ideia da separatriz:** imagine todos os municípios em fila, do mais pobre ao mais rico em PIB per capita. O **percentil 10** é o valor que fica perto da posição “10% da fila”.

- Cerca de **10%** dos municípios ficam **abaixo** desse valor.
- Cerca de **90%** ficam **iguais ou acima**.

Dez por cento de 167 municípios = 16,7. Por isso o grupo deve ter **cerca de 17** municípios.

A atividade pede o recorte estatístico (percentil), não um “top 17” escolhido no olho.


## 2.1 Calcular o percentil 10

O NumPy faz isso com `np.percentile(lista_de_valores, 10)`.

Ele **ordena** os números e **interpola** (faz uma média ponderada) entre os dois vizinhos da posição 10%. Depois arredondamos para 2 casas, porque PIB per capita está em reais e centavos.


In [ ]:
# dropna() remove células vazias, se houver, para o percentil não quebrar
valores_pib = tabela["pib_per_capita"].dropna()

# percentile(..., 10) = valor que deixa cerca de 10% dos municípios à esquerda
percentil_10 = round(float(np.percentile(valores_pib, 10)), 2)

print("Quantidade de municípios usados no cálculo:", len(valores_pib))
print("Percentil 10 (P10) = R$", br(percentil_10))
print()
print("Interpretação em uma frase:")
print("Cerca de 10% dos municípios do RN têm PIB per capita menor que R$", br(percentil_10) + ".")


### Resposta da pergunta 2.1

O **percentil 10** é o número impresso na célula anterior (rótulo **P10**).

Esse valor é a **separatriz**: daqui para frente, o grupo da pergunta 2.2 é “quem tem PIB per capita **menor que** esse número”.

Na coleta de referência deste projeto, o P10 ficou em torno de **R$ 13.668,37**. Se o IBGE revisar a série, o valor impresso acima é o que vale.


## 2.2 Separar os municípios com PIB per capita menor que o percentil 10

**Regra (lida ao pé da letra da atividade):**

```text
entra no grupo se  PIB per capita  <  percentil 10
```

O símbolo `<` significa **estritamente menor**. Quem tiver exatamente o valor do P10 **não entra**.


In [ ]:
# True = o município entra no grupo da pergunta 2.2
tabela["abaixo_do_p10"] = tabela["pib_per_capita"] < percentil_10

# Os colchetes filtram: ficam só as linhas em que a condição é True
grupo_p10 = tabela[tabela["abaixo_do_p10"]].copy()

# Ordena do menor para o maior PIB per capita
grupo_p10 = grupo_p10.sort_values("pib_per_capita")

print("Regra aplicada: pib_per_capita <", br(percentil_10))
print("Municípios no grupo:", len(grupo_p10))
print("Isso equivale a", br(100 * len(grupo_p10) / len(tabela), 1), "% dos municípios do RN.")
print()
print("Nomes do grupo:")
for i, nome in enumerate(grupo_p10["municipio"], start=1):
    print(f"  {i:2d}. {nome}")


### Resposta da pergunta 2.2

Tabela do grupo (município, PIB per capita, população e mesorregião):


In [ ]:
# Só as colunas pedidas na pergunta 2.2 (nome, território, PIB e população)
grupo_p10[
    [
        "municipio",
        "mesorregiao",
        "microrregiao",
        "pib_per_capita",
        "populacao_censo_2022",
        "densidade_demografica",
    ]
]


## 2.3 Variáveis que podem explicar esses PIBs per capita

O PIB per capita é uma **razão**: produção do município ÷ população. Um valor baixo pode aparecer quando:

- a economia é **pequena** e pouco diversificada;
- quase tudo que se produz vem da **administração pública** (prefeitura, saúde, educação, transferências);
- há **pouca indústria**;
- o município é **pequeno e pouco denso**;
- os indicadores de **educação e desenvolvimento humano** são mais fracos.

Por isso comparamos o **grupo P10** com os **demais municípios do RN**. Se uma diferença aparece com clareza nos dois grupos, ela ajuda a explicar o recorte (não “prova” uma única causa).

| Variável | Por que entra na análise |
| --- | --- |
| População e densidade | Municípios menores tendem a ter menos atividade econômica |
| Área | Porte físico e dispersão, junto com a densidade |
| % administração pública no VAB | Dependência de salários e serviços públicos |
| % indústria e serviços | Sinal de diversificação produtiva |
| % agropecuária | Peso do setor primário |
| Alfabetização e escolarização | Escolaridade ligada à produtividade |
| IDHM | Síntese de renda, educação e longevidade |
| Salário médio e população ocupada | Mercado de trabalho formal |


In [ ]:
# Os "demais" são todos os que NÃO estão abaixo do P10
# O símbolo ~ inverte verdadeiro/falso: True vira False e vice-versa
demais = tabela[~tabela["abaixo_do_p10"]].copy()

print("Municípios no grupo P10:", len(grupo_p10))
print("Demais municípios do RN:", len(demais))


In [ ]:
# Cada item: (nome da coluna na tabela, rótulo para a professora ler, casas decimais)
variaveis = [
    ("pib_per_capita", "PIB per capita (R$)", 2),
    ("pib_mil_reais", "PIB total (R$ mil)", 2),
    ("populacao_censo_2022", "População (Censo 2022)", 0),
    ("densidade_demografica", "Densidade (hab/km²)", 1),
    ("area_km2", "Área (km²)", 1),
    ("participacao_agropecuaria_pct", "Participação da agropecuária no VAB (%)", 1),
    ("participacao_industria_pct", "Participação da indústria no VAB (%)", 1),
    ("participacao_servicos_pct", "Participação dos serviços no VAB (%)", 1),
    ("participacao_adm_publica_pct", "Participação da administração pública no VAB (%)", 1),
    ("taxa_alfabetizacao_15_mais_pct", "Alfabetização 15 anos ou mais (%)", 1),
    ("idhm", "IDHM", 3),
    ("taxa_escolarizacao_6_a_14_pct", "Escolarização 6 a 14 anos (%)", 1),
    ("salario_medio_mensal", "Salário médio (salários mínimos)", 2),
    ("populacao_ocupada", "População ocupada (%)", 1),
]

linhas_comparativo = []  # vai virar uma tabela: uma linha por variável
for coluna, rotulo, casas in variaveis:
    # mean() = média aritmética de todos os municípios daquele grupo
    media_grupo = grupo_p10[coluna].mean()
    media_demais = demais[coluna].mean()
    linhas_comparativo.append(
        {
            "Variável": rotulo,
            "Média do grupo P10": br(media_grupo, casas),
            "Média dos demais municípios": br(media_demais, casas),
            # só um rótulo de leitura: o grupo P10 está abaixo ou acima da média do resto?
            "O grupo P10 fica": "abaixo" if media_grupo < media_demais else "acima",
        }
    )

comparativo = pd.DataFrame(linhas_comparativo)
comparativo


### Como ler a tabela comparativa

- Se o grupo P10 tem **menos população**, **menos indústria** e **mais administração pública**, isso descreve economias pequenas e dependentes do setor público.
- A alfabetização e o IDHM um pouco **menores** no grupo P10 reforçam um quadro social mais frágil.
- Isso **não** diz que “falta indústria causa PIB baixo” sozinha. Diz que, **em conjunto**, esses municípios compartilham um perfil parecido.

Agora os números de cada município do grupo, para a questão 2.3 ficar completa:


In [ ]:
# Recorte das variáveis da pergunta 2.3, município a município (não é média)
grupo_p10[
    [
        "municipio",
        "pib_per_capita",
        "populacao_censo_2022",
        "densidade_demografica",
        "participacao_agropecuaria_pct",
        "participacao_industria_pct",
        "participacao_servicos_pct",
        "participacao_adm_publica_pct",
        "taxa_alfabetizacao_15_mais_pct",
        "idhm",
        "populacao_ocupada",
    ]
]


### Gráfico 1: onde o P10 corta a distribuição

Cada barra do histograma conta **quantos municípios** caem naquela faixa de PIB per capita. A linha vermelha é o percentil 10.


In [ ]:
import matplotlib.pyplot as plt  # biblioteca de gráficos

plt.figure(figsize=(9, 4))
# histograma: cada barra = quantos municípios naquela faixa de PIB per capita
plt.hist(valores_pib, bins=25, color="#4F81BD", edgecolor="white")
# linha vermelha = percentil 10 (a separatriz)
plt.axvline(
    percentil_10,
    color="#C00000",
    linestyle="--",
    linewidth=2,
    label=f"P10 = R$ {br(percentil_10)}",
)
# corta a cauda extrema à direita só para o desenho ficar legível
plt.xlim(0, valores_pib.quantile(0.95))
plt.xlabel("PIB per capita 2023 (R$)")
plt.ylabel("Número de municípios")
plt.title("Distribuição do PIB per capita municipal no RN")
plt.legend()
plt.tight_layout()
plt.show()


### Gráfico 2: estrutura produtiva (média do VAB)

Compara o peso de cada setor no grupo P10 e nos demais municípios. O valor adicionado é de **2021**, último ano em que o IBGE publicou a composição setorial completa.


In [ ]:
# Quatro setores (percentual do valor adicionado bruto em 2021)
setores = [
    "participacao_agropecuaria_pct",
    "participacao_industria_pct",
    "participacao_servicos_pct",
    "participacao_adm_publica_pct",
]
nomes = ["Agropecuária", "Indústria", "Serviços", "Adm. pública"]

media_grupo = [grupo_p10[c].mean() for c in setores]
media_resto = [demais[c].mean() for c in setores]

posicoes = range(len(nomes))
plt.figure(figsize=(9, 4))
# vermelho = grupo P10; azul = demais municípios do RN
plt.bar([p - 0.18 for p in posicoes], media_grupo, width=0.36, label="Grupo abaixo do P10", color="#C00000")
plt.bar([p + 0.18 for p in posicoes], media_resto, width=0.36, label="Demais municípios", color="#4F81BD")
plt.xticks(list(posicoes), nomes)
plt.ylabel("% do valor adicionado bruto (2021)")
plt.title("Estrutura produtiva média: grupo P10 versus demais municípios")
plt.legend()
plt.tight_layout()
plt.show()


---
# Descrição sumária da situação

Esta seção junta as respostas 1, 2.1, 2.2 e 2.3 em um parágrafo. Os números vêm das células anteriores, não foram digitados à mão.


In [ ]:
n = len(grupo_p10)  # quantos municípios ficaram abaixo do P10
nomes = ", ".join(grupo_p10["municipio"].tolist())  # lista os nomes em uma frase

# value_counts() conta quantos municípios do grupo há em cada mesorregião
meso = grupo_p10["mesorregiao"].value_counts().head(3)
meso_txt = "; ".join(f"{nome} ({qtd})" for nome, qtd in meso.items())

# Texto montado com os números já calculados (não é digitado à mão)
descricao = (
    f"O PIB per capita de 2023 dos {len(tabela)} municípios do Rio Grande do Norte "
    f"tem percentil 10 igual a R$ {br(percentil_10)}. Ficaram abaixo dessa separatriz "
    f"{n} municípios: {nomes}. "
    f"Nesse grupo, a média do PIB per capita é R$ {br(grupo_p10['pib_per_capita'].mean())}, "
    f"frente a R$ {br(tabela['pib_per_capita'].mean())} na média estadual. "
    f"São, em geral, municípios pequenos (população média de "
    f"{br(grupo_p10['populacao_censo_2022'].mean(), 0)} habitantes, contra "
    f"{br(demais['populacao_censo_2022'].mean(), 0)} nos demais) e de baixa densidade "
    f"({br(grupo_p10['densidade_demografica'].mean(), 1)} hab/km² contra "
    f"{br(demais['densidade_demografica'].mean(), 1)}). "
    f"A estrutura produtiva de 2021 indica maior dependência da administração pública "
    f"({br(grupo_p10['participacao_adm_publica_pct'].mean(), 1)}% do VAB no grupo P10 "
    f"contra {br(demais['participacao_adm_publica_pct'].mean(), 1)}% nos demais) e "
    f"menor peso da indústria ({br(grupo_p10['participacao_industria_pct'].mean(), 1)}% "
    f"contra {br(demais['participacao_industria_pct'].mean(), 1)}%). "
    f"Há também pior desempenho educacional e de desenvolvimento humano: taxa de "
    f"alfabetização de 15 anos ou mais de "
    f"{br(grupo_p10['taxa_alfabetizacao_15_mais_pct'].mean(), 1)}% "
    f"(contra {br(demais['taxa_alfabetizacao_15_mais_pct'].mean(), 1)}%) "
    f"e IDHM médio de {br(grupo_p10['idhm'].mean(), 3)} "
    f"(contra {br(demais['idhm'].mean(), 3)}). "
    f"A concentração territorial do grupo está principalmente em: {meso_txt}. "
    f"Em conjunto, o recorte aponta municípios de pequeno porte, com economia pouco "
    f"diversificada, forte peso do setor público e indicadores sociais mais frágeis, "
    f"o que ajuda a explicar o PIB per capita situado no décimo inferior da distribuição estadual."
)

print(descricao)


## Exportar a planilha da atividade

A célula abaixo grava um Excel com:

1. todos os municípios;
2. o grupo abaixo do P10;
3. o comparativo de médias.


In [ ]:
# Cria a pasta data/ se ela ainda não existir
pasta = raiz / "data"
pasta.mkdir(exist_ok=True)
arquivo_xlsx = pasta / "atividade_separatrizes_pib_rn.xlsx"

# index=False evita uma coluna extra 0, 1, 2... (número da linha do pandas)
with pd.ExcelWriter(arquivo_xlsx, engine="openpyxl") as excel:
    pib_todos.to_excel(excel, sheet_name="1_PIB_todos_municipios", index=False)
    grupo_p10.to_excel(excel, sheet_name="2.2_Grupo_abaixo_P10", index=False)
    comparativo.to_excel(excel, sheet_name="2.3_Comparativo", index=False)

print("Planilha da atividade salva em:", arquivo_xlsx)
print("No Colab: use a pasta à esquerda da tela para baixar o arquivo.")


---
# Recado final

| Pergunta | O que este notebook fez |
| --- | --- |
| **1** | Listou o PIB per capita dos 167 municípios (IBGE, 2023) |
| **2.1** | Calculou o percentil 10 com `numpy.percentile` |
| **2.2** | Filtrou quem tem PIB per capita **menor que** o P10 |
| **2.3** | Comparou população, estrutura produtiva, alfabetização e IDHM |
| **Descrição** | Juntou esses números em um parágrafo |

Para **gerar de novo** os CSV/Excel a partir das APIs:

- notebook: `gerar_dataset.ipynb`
- ou no terminal: `python gerar_dataset.py`

Detalhes das URLs do IBGE e dos arquivos Python: `README.md`.
